# Sorted GCD Pair Queries

You are given an integer array `nums` of length `n` and an integer array `queries`.

Let `gcdPairs` denote an array obtained by calculating the GCD of all possible pairs `(nums[i], nums[j])`, where `0 <= i < j < n`, and then sorting these values in **ascending** order.

For each query `queries[i]`, you need to find the element at index `queries[i]` in `gcdPairs`.

Return an integer array `answer`, where `answer[i]` is the value at `gcdPairs[queries[i]]` for each query.

The term `gcd(a, b)` denotes the **greatest common divisor** of `a` and `b`.

### Example 1:

**Input:** nums = [2,3,4], queries = [0,2,2]

**Output:** [1,2,2]

**Explanation:**

`gcdPairs = [gcd(nums[0], nums[1]), gcd(nums[0], nums[2]), gcd(nums[1], nums[2])] = [1, 2, 1]`.

After sorting in ascending order, `gcdPairs = [1, 1, 2]`.

So, the answer is `[gcdPairs[queries[0]], gcdPairs[queries[1]], gcdPairs[queries[2]]] = [1, 2, 2]`.

### Example 2:

**Input:** nums = [4,4,2,1], queries = [5,3,1,0]

**Output:** [4,2,1,1]

**Explanation:**

`gcdPairs` sorted in ascending order is `[1, 1, 1, 2, 2, 4]`.

### Example 3:

**Input:** nums = [2,2], queries = [0,0]

**Output:** [2,2]

**Explanation:**

`gcdPairs = [2]`.

### Constraints:

*   `2 <= n == nums.length <= 105`
*   `1 <= nums[i] <= 5 * 104`
*   `1 <= queries.length <= 105`
*   `0 <= queries[i] < n * (n - 1) / 2`

In [ ]:
from math import gcd

class Solution:
    def gcdValues(self, nums: list[int], queries: list[int]) -> list[int]:
        gcd_pairs: list = []
        n = len(nums)
        for i in range(n):
            for j in range(i+1, n):
                gcd_pairs.append(gcd(nums[i], nums[j]))
                
        gcd_pairs.sort()
        
        output = [gcd_pairs[num] for num in queries]
        return output
        

## Optimized Approach — O(M log M + q log M)

Instead of enumerating all O(n²) pairs, we use **frequency counting + inclusion-exclusion**.

### Step 1 — Frequency count
Count how many times each value appears in `nums` using a frequency array `cnt` of size `M+1` (where `M = max(nums)`).

### Step 2 — Count pairs with GCD divisible by g (for every g)
For each candidate GCD value `g` from 1 to M:
- Sum up `cnt[g] + cnt[2g] + cnt[3g] + ...` → call this `total_multiples`.
- The number of pairs whose GCD is **divisible** by `g` = `C(total_multiples, 2) = total_multiples * (total_multiples - 1) // 2`.
- Store this in `div_pairs[g]`.

This is essentially a sieve and runs in O(M log M) (harmonic series).

### Step 3 — Inclusion-exclusion to get exact GCD count
Iterate `g` from M down to 1.  
`exact[g]` = pairs with GCD **exactly** `g` = `div_pairs[g]` minus the count of pairs whose GCD is a proper multiple of `g`:
$$\text{exact}[g] = \text{div\_pairs}[g] - \sum_{k=2}^{\lfloor M/g \rfloor} \text{exact}[k \cdot g]$$

### Step 4 — Build a sorted prefix-sum array
Since GCDs range 1..M, we can build:
- `gcd_vals`: all distinct GCD values that appear (sorted ascending).
- `prefix`: cumulative count of pairs up to each value.

This represents the `gcdPairs` array without materializing it.

### Step 5 — Answer queries with binary search
For each `query[i]`, binary search in `prefix` to find which GCD value falls at that index → O(log M) per query.

In [ ]:
from bisect import bisect_right

class SolutionOptimized:
    def gcdValues(self, nums: list[int], queries: list[int]) -> list[int]:
        max_val = max(nums)

        # Step 1: count frequency of each value in nums
        value_count = [0] * (max_val + 1)
        for num in nums:
            value_count[num] += 1

        # Step 2: for each gcd_candidate g, count pairs whose GCD is divisible by g
        # sum all value_count[g], value_count[2g], value_count[3g], ...
        # then C(total, 2) gives pairs divisible by g
        divisible_pair_count = [0] * (max_val + 1)
        for gcd_candidate in range(1, max_val + 1):
            multiples_total = sum(value_count[gcd_candidate::gcd_candidate])
            divisible_pair_count[gcd_candidate] = multiples_total * (multiples_total - 1) // 2

        # Step 3: inclusion-exclusion (high → low) to isolate pairs with GCD exactly g
        exact_gcd_count = [0] * (max_val + 1)
        for gcd_candidate in range(max_val, 0, -1):
            exact_gcd_count[gcd_candidate] = divisible_pair_count[gcd_candidate]
            for next_multiple in range(2 * gcd_candidate, max_val + 1, gcd_candidate):
                exact_gcd_count[gcd_candidate] -= exact_gcd_count[next_multiple]

        # Step 4: build prefix-sum over sorted distinct GCD values
        # (represents the virtual sorted gcdPairs array without materializing it)
        distinct_gcds = []
        cumulative_counts = []
        running_total = 0
        for gcd_candidate in range(1, max_val + 1):
            if exact_gcd_count[gcd_candidate] > 0:
                running_total += exact_gcd_count[gcd_candidate]
                distinct_gcds.append(gcd_candidate)
                cumulative_counts.append(running_total)

        # Step 5: answer each query with binary search into the prefix-sum
        return [distinct_gcds[bisect_right(cumulative_counts, query_index)] for query_index in queries]


# --- Tests ---
sol = SolutionOptimized()
print(sol.gcdValues([2, 3, 4], [0, 2, 2]))          # [1, 2, 2]
print(sol.gcdValues([4, 4, 2, 1], [5, 3, 1, 0]))    # [4, 2, 1, 1]
print(sol.gcdValues([2, 2], [0, 0]))                 # [2, 2]


[PY] 3312-sorted-gcd-pair-queries.ipynb:5 - M: 4
[PY] 3312-sorted-gcd-pair-queries.ipynb:9 - cnt: [0, 0, 0, 0, 0]
[PY] 3312-sorted-gcd-pair-queries.ipynb:12 - cnt: [0, 1, 1, 0, 2]
[PY] 3312-sorted-gcd-pair-queries.ipynb:16 - div_pairs: [0, 0, 0, 0, 0]
[PY] 3312-sorted-gcd-pair-queries.ipynb:20 - div_pairs: [0, 6, 3, 0, 1]
[PY] 3312-sorted-gcd-pair-queries.ipynb:24 - exact: [0, 0, 0, 0, 0]
[PY] 3312-sorted-gcd-pair-queries.ipynb:29 - exact: [0, 3, 2, 0, 1]
[PY] 3312-sorted-gcd-pair-queries.ipynb:43 - output: [4, 2, 1, 1]
None
